In [1]:
## Definition of the DrugSDA SCP server, including basic operations such as connect, disconnect, list_tools, and parse_result.
import asyncio
import json
from mcp.client.streamable_http import streamablehttp_client
from mcp import ClientSession

DrugSDA_Tool_SERVER_URL = "https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool"      ## DrugSDA-Tool Server
# DrugSDA_Tool_SERVER_URL = "http://180.184.86.2:32208/mcp"

class DrugSDAClient:    
    def __init__(self, server_url: str):
        self.server_url = server_url
        self.session = None
        
    async def connect(self):
        print(f"server url: {self.server_url}")
        try:
            self.transport = streamablehttp_client(
                url=self.server_url,
                headers={"SCP-HUB-API-KEY": "REDACTED_MOLCLAW_KEY"}
            )
            self.read, self.write, self.get_session_id = await self.transport.__aenter__()
            
            self.session_ctx = ClientSession(self.read, self.write)
            self.session = await self.session_ctx.__aenter__()

            await self.session.initialize()
            session_id = self.get_session_id()
            
            print(f"✓ connect success")
            return True
            
        except Exception as e:
            print(f"✗ connect failure: {e}")
            import traceback
            traceback.print_exc()
            return False
    
    async def disconnect(self):
        try:
            if self.session:
                await self.session_ctx.__aexit__(None, None, None)
            if hasattr(self, 'transport'):
                await self.transport.__aexit__(None, None, None)
            print("✓ already disconnect")
        except Exception as e:
            print(f"✗ disconnect error: {e}")
    
    async def list_tools(self):        
        try:
            tools_list = await self.session.list_tools()
            print(f"tool count: {len(tools_list.tools)}")
            
            for i, tool in enumerate(tools_list.tools, 1):
                print(f"{i:2d}. {tool.name}")
                if tool.description:
                    desc_line = tool.description.split('\n')[0]
                    print(f"    {desc_line}")
            
            print(f"✓ Get tool list success")
            return tools_list.tools
            
        except Exception as e:
            print(f"✗ Get tool list fail: {e}")
            return []
    
    def parse_result(self, result):
        try:
            if hasattr(result, 'content') and result.content:
                content = result.content[0]
                if hasattr(content, 'text'):
                    return json.loads(content.text)
            return str(result)
        except Exception as e:
            return {"error": f"parse error: {e}", "raw": str(result)}

In [3]:
## pred_protein_structure_esmfold
async def main():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    ## Input protein sequence
    # sequence = "MSKGEELFTGVVPILVELDGDVNGHKFSVSGEGEGDATYGKLTLKFICTTGKLPVPWPTLVTTLTYGVQCFSRYPDHMKQHDFFKSAMPEGYVQERTIFFKDDGNYKTRAEVKFEGDTLVVRIELKGIDFKEDGNILGHKLEYNYNSHNVYITADKQKNGIKANFKIRHNIEDGSVQLADHYQQNTPIGDGPVLLPDNHYLSTQSALSKDPNEKRDHMVLLEFVTAAGITLGMDELYK"
    input_path = '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/protein.pdb'

    result = await client.session.call_tool(
        "fix_pdb",
        arguments={
            "input_path": input_path
        }
    )
    
    result_data = client.parse_result(result)
    print (result_data)
    
    await client.disconnect()

if __name__ == '__main__':
    await main()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool


✓ connect success
{'status': 'success', 'msg': 'PDB repair finished', 'atom_count': 1926, 'residue_count': 234, 'chain_count': 1, 'output_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/pdbfixer_result/pdbfixer_20260325_010538_5387f4', 'output_file': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/pdbfixer_result/pdbfixer_20260325_010538_5387f4/protein_fixed.pdb'}
✓ already disconnect


In [3]:
async def test_chai1_predict():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    # 对应源命令:
    # python chai1_predict.py --seq "MKFLILLFNILCLFPVLAADNHGVS" --name my_protein --dry-run
    
    result = await client.session.call_tool(
        "run_chai1_prediction",
        arguments={
            "mode": "sequence",
            "seq": "MKFLILLFNILCLFPVLAADNHGVS",
            "name": "my_protein",
            "samples": 5,
            "dry_run": True
        }
    )
    
    result_data = client.parse_result(result)
    print("Chai-1 Result:", result_data)
    
    await client.disconnect()

# 运行测试
await test_chai1_predict()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
Chai-1 Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: run_chai1_prediction', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [4]:
async def test_diffdock_auto():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    # 对应源命令参数:
    # --protein_path ~/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/outputok.pdb
    # --ligand "CC(C)Cc1ccc(C)cc1"
    
    result = await client.session.call_tool(
        "run_diffdock_auto",
        arguments={
            "protein_path": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/outputok.pdb",
            "ligand": "CC(C)Cc1ccc(C)cc1",
            # 其他参数使用默认值: inference_steps=20, samples_per_complex=40, device="cuda" 等
        }
    )
    
    result_data = client.parse_result(result)
    print("DiffDock Result:", result_data)
    
    await client.disconnect()

# 运行测试
if __name__ == '__main__':
    await test_diffdock_auto()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
DiffDock Result: {'status': 'success', 'msg': 'completed with 1 complexes', 'output_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/diffdoc_result/diffdock_20260303_140416_de62c2', 'summary_metrics': {'num_complexes': 1, 'best_confidence': -1.42, 'average_confidence': -1.42, 'median_confidence': -1.42}, 'quality_distribution': {'良好 (Good)': 1}, 'complex_results': {'complex_0': {'complex_name': 'complex_0', 'num_poses': 10, 'best_confidence': -1.42, 'best_pose_file': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/diffdoc_result/diffdock_20260303_140416_de62c2/complex_0/rank10_confidence-1.42.sdf', 'mean_confidence': -0.9039999999999999, 'std_confidence': 0.3226825064982606, 'poses': [{'rank': 10, 'confidence': -1.42, 'file': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/diffdoc_result/diffdock_20260303_140416_de62c2/complex_0/rank10_confidence-1.42.sdf'}, {'rank': 9, 

In [3]:
async def test_equiscore_extract():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    # 对应源命令参数映射
    result = await client.session.call_tool(
        "equiscore_extract_pocket",
        arguments={
            "docking_result": "tool_src_fzl/data/ligand.sdf",
            "receptor_pdb": "tool_src/hdock_tool/receptor.pdb",
            # single_sdf_save_path 和 pocket_save_dir 通常由后端自动管理，
            # 若需指定，请检查工具定义是否支持 output_dir 类似参数
            "dry_run": False
        }
    )
    
    result_data = client.parse_result(result)
    print("EquiScore Extract Result:", result_data)
    
    await client.disconnect()


await test_equiscore_extract()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
EquiScore Extract Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: equiscore_extract_pocket', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [6]:
async def test_evobind_design():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "design_peptide_binder_evobind",
        arguments={
            "fasta": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/data/1ssc_receptor.fasta",
            "peptide_length": 5,
            "num_designs": 1,
            # 其他参数使用默认值: num_iterations=100, model_name="model_1", cyclic=False 等
            "dry_run": False
        }
    )
    
    result_data = client.parse_result(result)
    print("EvoBind Result:", result_data)
    
    await client.disconnect()

await test_evobind_design()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
EvoBind Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: design_peptide_binder_evobind', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [7]:
async def test_hdock_docking():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "run_hdock_docking",
        arguments={
            "receptor": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src/hdock_tool/receptor.pdb",
            "ligand": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src/hdock_tool/ligand.pdb",
            "nmax": 10,
            "angle": 15,
            # no_complex, rsite, lsite 使用默认值
        }
    )
    
    result_data = client.parse_result(result)
    print("HDOCK Result:", result_data)
    
    await client.disconnect()

await test_hdock_docking()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
HDOCK Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: run_hdock_docking', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [8]:
async def test_openawsem_sim():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "run_openawsem_simulation",
        arguments={
            "pdb": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/protein_fixed.pdb",
            "mode": "nvt",
            "temperature": 300.0,
            "steps": 50000.0,  # 5e4
            "platform": "CUDA",
            # 其他参数使用默认值: sim_dir=None, use_frag_mem=False, gpu_id="0"
            "dry_run": False
        }
    )
    
    result_data = client.parse_result(result)
    print("OpenAWSEM Result:", result_data)
    
    await client.disconnect()

await test_openawsem_sim()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
OpenAWSEM Result: {'status': 'success', 'msg': 'simulation completed', 'output_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openawsem_sim_result/openawsem_sim_20260303_140512_de8669b1', 'simulation_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openawsem_sim_result/openawsem_sim_20260303_140512_de8669b1/protein_fixed', 'steps': 50000.0, 'mode': 'nvt', 'temperature': 300.0, 'platform': 'CUDA', 'dry_run': False, 'output_files': {'final_pdb': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openawsem_sim_result/openawsem_sim_20260303_140512_de8669b1/protein_fixed/protein_fixed_final.pdb', 'energy_log': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openawsem_sim_result/openawsem_sim_20260303_140512_de8669b1/protein_fixed/protein_fixed_energy.log', 'checkpoint': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openawsem_sim_result/openawsem_sim_20

In [9]:
async def test_prolif_docking():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "prolif_analyze_docking",
        arguments={
            "protein_path": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/1azm/1azm_protein.pdb",
            "ligand_paths": ["/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/1azm/1azm_ligand.sdf"],
            "ligand_format": "sdf",
            # 其他参数使用默认值: template_smiles=None, count=False
        }
    )
    
    result_data = client.parse_result(result)
    print("ProLIF Docking Result:", result_data)
    
    await client.disconnect()

await test_prolif_docking()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
ProLIF Docking Result: {'status': 'success', 'msg': 'completed', 'command': 'docking', 'output_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/prolif_result/docking_20260303_140540_9217b8', 'output_file': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/prolif_result/docking_20260303_140540_9217b8/docking_results.csv', 'n_frames': 1, 'n_interactions': 7, 'frequent_interactions': [{'interaction': "('UNL1', 'ZN1', 'VdWContact')", 'frequency': 1.0}, {'interaction': "('UNL1', 'HOH57', 'HBDonor')", 'frequency': 1.0}, {'interaction': "('UNL1', 'HOH57', 'VdWContact')", 'frequency': 1.0}, {'interaction': "('UNL1', 'HOH99', 'VdWContact')", 'frequency': 1.0}, {'interaction': "('UNL1', 'PHE91.A', 'VdWContact')", 'frequency': 1.0}, {'interaction': "('UNL1', 'HIS94.A', 'HBDonor')", 'frequency': 1.0}, {'interaction': "('UNL1', 'HIS94.A', 'VdWContact')", 'frequency': 1.0}], 'result_summary': {'

In [10]:
async def test_openmm_md():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    # 注意：源命令中的时间单位可能是 ns，而 MCP 工具定义通常为 ps。
    # 如果源命令 --md-time 5 代表 5ns，则下面应为 5000.0。此处暂按原数值 5.0 传递测试。
    result = await client.session.call_tool(
        "run_openmm_protein_md",
        arguments={
            "protein_pdb": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/protein_fixed.pdb",
            "solvent_type": "implicit",
            "gb_model": "GBn2",
            "water_model": "tip3p", # 默认值，隐式溶剂下可能不使用，但需传入
            "force_field": "amber14", # 默认值
            "md_time": 5.0,  # 请确认单位：如果是 5ns，请改为 5000.0
            "platform": "CUDA", # 源命令未指定，默认 CUDA 或 CPU，根据环境调整
            "full_md": True,
            "nvt_time": 1000.0, # 假设源命令 1 为 1ns -> 1000ps
            "npt_time": 1000.0  # 假设源命令 1 为 1ns -> 1000ps
        }
    )
    
    result_data = client.parse_result(result)
    print("OpenMM MD Result:", result_data)
    
    await client.disconnect()

await test_openmm_md()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
OpenMM MD Result: {'error': 'parse error: Extra data: line 1 column 3 (char 2)', 'raw': "meta=None content=[TextContent(type='text', text='2 validation errors for call[run_openmm_protein_md]\\nnvt_time\\n  Unexpected keyword argument [type=unexpected_keyword_argument, input_value=1000.0, input_type=float]\\n    For further information visit https://errors.pydantic.dev/2.12/v/unexpected_keyword_argument\\nnpt_time\\n  Unexpected keyword argument [type=unexpected_keyword_argument, input_value=1000.0, input_type=float]\\n    For further information visit https://errors.pydantic.dev/2.12/v/unexpected_keyword_argument', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [11]:
async def test_proteinmpnn_design():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "run_proteinmpnn_design",
        arguments={
            "pdb_input": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/outputok.pdb",
            "num_seq": 8,
            "sampling_temp": "0.1",
            # 其他参数使用默认值: model_name="v_48_020", chains_to_design="" 等
            "dry_run": False
        }
    )
    
    result_data = client.parse_result(result)
    print("ProteinMPNN Result:", result_data)
    
    await client.disconnect()

await test_proteinmpnn_design()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
ProteinMPNN Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: run_proteinmpnn_design', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [12]:
async def test_bioemu_sampling():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "run_bioemu_sampling",
        arguments={
            "sequence": "GYDPETGTWG",
            "num_samples": 5,
            "export_pdbs": False, # 默认
            "dry_run": False
        }
    )
    
    result_data = client.parse_result(result)
    print("BioEmu Result:", result_data)
    
    await client.disconnect()

await test_bioemu_sampling()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
BioEmu Result: {'status': 'success', 'msg': 'BioEmu sampling completed', 'command': 'run_bioemu', 'run_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/bioemu_result/bioemu_output_20260303T060603Z_950fa3', 'output_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/bioemu_result/bioemu_output_20260303T060603Z_950fa3', 'files': ['/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/bioemu_result/bioemu_output_20260303T060603Z_950fa3/batch_0000000_0000005.npz', '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/bioemu_result/bioemu_output_20260303T060603Z_950fa3/samples.xtc', '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/bioemu_result/bioemu_output_20260303T060603Z_950fa3/sampling_statistics.json', '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/bioemu_result/bioemu_output_20260303T060603Z_950fa3/sequence.fasta', '/root/lwj/wll/code/DrugAgentTools/sxy

In [1]:
async def test_mmpbsa_auto():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "run_mmpbsa_calculation",
        arguments={
            "work_dir": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/gmx_mmpbsa_result/20260227T070917Z_2bc876/protein_fixed_ligand_pose1",
            "method": "both",
            "nproc": 32,
            "interval": 1,
            "startframe": 1,          # 默认值
            "endframe": 10000000001,  # 默认值
            "generate_input": True,
            "dry_run": False
        }
    )
    
    result_data = client.parse_result(result)
    print("MMPBSA Auto Result:", result_data)
    
    await client.disconnect()

await test_mmpbsa_auto()

NameError: name 'DrugSDAClient' is not defined

In [14]:
async def test_mmpbsa_propro():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "run_mmpbsa_propro",
        arguments={
            "work_dir": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/gmx_mmpbsa_result/protein2_fixed_MD_20260227_190758",
            "method": "gb",
            "nproc": 64,
            "skip_mmpbsa": False,
            "gro": "em.gro",
            "dry_run": False
        }
    )
    
    result_data = client.parse_result(result)
    print("MMPBSA ProPro Result:", result_data)
    
    await client.disconnect()

await test_mmpbsa_propro()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
MMPBSA ProPro Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: run_mmpbsa_propro', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [15]:
async def test_prepare_complex():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "prepare_protein_ligand_complex",
        arguments={
            "protein": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/protein_fixed.pdb",
            "ligand": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/ligand.sdf",
            "pose": 1,             # 默认值
            "gpu_ids": "0",        # 默认值
            "full_md": True,
            "nvt_time": 1.0,       # 默认值 (ps)
            "npt_time": 1.0,       # 默认值 (ps)
            "md_time": 100.0,      # 默认值 (ps)
            "ph": None
        }
    )
    
    result_data = client.parse_result(result)
    print("Prepare Complex Result:", result_data)
    
    await client.disconnect()

await test_prepare_complex()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
Prepare Complex Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: prepare_protein_ligand_complex', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [ ]:
async def test_mmpbsa_analyzer():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    result = await client.session.call_tool(
        "analyze_mmpbsa",
        arguments={
            "work_dir": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/gmx_mmpbsa_result/20260227T070917Z_2bc876/protein_fixed_ligand_pose1",
            "decomp": None,
            "results": None,
            "method": None,
            "pb_decomp": None,
            "pb_results": None,
            "gb_decomp": None,
            "gb_results": None,
            "gro": None
        }
    )
    
    result_data = client.parse_result(result)
    print("MMPBSA Analyzer Result:", result_data)
    
    await client.disconnect()

await test_mmpbsa_analyzer()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
MMPBSA Analyzer Result: {'error': 'parse error: Expecting value: line 1 column 1 (char 0)', 'raw': "meta=None content=[TextContent(type='text', text='Unknown tool: analyze_mmpbsa_results', annotations=None, meta=None)] structuredContent=None isError=True"}
✓ already disconnect


In [18]:
async def test_openmm_md():
    client = DrugSDAClient(DrugSDA_Tool_SERVER_URL)
    if not await client.connect():
        print("connection failed")
        return
    
    # 注意：源命令中的时间单位通常是 ns，而 MCP 工具定义中明确为 ps。
    # 如果源命令 --md-time 5 代表 5ns，则下面应为 5000.0。
    # 此处暂按源命令字面数值 5.0 传递，若需 5ns 请改为 5000.0。
    result = await client.session.call_tool(
        "run_openmm_protein_md",
        arguments={
            "protein_pdb": "/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_src_fzl/test_dir/protein_fixed.pdb",
            "solvent_type": "implicit",
            "gb_model": "GBn2",
            "water_model": "tip3p",      # 默认值
            "force_field": "amber14",    # 默认值
            "md_time": 5.0,              # ⚠️ 确认单位：若是 5ns 请改为 5000.0
            "platform": "CUDA",          # 默认值
            "full_md": True,
            # 注意：run_openmm_protein_md 工具定义中没有 nvt_time/npt_time 参数，它们包含在 full_md 流程内部或使用默认值。
            # 如果源脚本的这些参数需要显式传递，请检查工具定义是否遗漏。
            # 根据工具51定义，只有 md_time, solvent_type 等，没有 nvt/npt 单独参数。
        }
    )
    
    result_data = client.parse_result(result)
    print("OpenMM MD Result:", result_data)
    
    await client.disconnect()

await test_openmm_md()

server url: https://scp.intern-ai.org.cn/api/v1/mcp/2/DrugSDA-Tool
✓ connect success
OpenMM MD Result: {'status': 'success', 'msg': 'MD run completed or initialized', 'command': 'protein_openmm_md', 'run_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openmm_md_result/protein_fixed_implicit_GBn2_20260303_141105', 'work_dir': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openmm_md_result/protein_fixed_implicit_GBn2_20260303_141105', 'md_time': 5.0, 'solvent_type': 'implicit', 'force_field': 'amber14', 'full_md': True, 'trajectory_path': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openmm_md_result/protein_fixed_implicit_GBn2_20260303_141105/md_traj.dcd', 'energy_log': '/root/lwj/wll/code/DrugAgentTools/sxy_local/tool_result/openmm_md_result/protein_fixed_implicit_GBn2_20260303_141105/md.log', 'generated_files': ['em.chk', 'em.pdb', 'md.log', 'md_final.chk', 'md_final.pdb', 'md_traj.dcd', 'nvt.chk', 'nvt.log', 'nvt.pdb', 'protein_fixed.pdb', 'protein